In [ ]:
"""
Parse the "Floating-point operations per clock cycle for various processors" table
from Wikipedia and save the data to a CSV file.
"""

import requests
from bs4 import BeautifulSoup
import csv
import re

def parse_wiki_flops_table():
    """
    Parse the "Floating-point operations per clock cycle for various processors" table
    from the Wikipedia page and save the data to a CSV file.
    """

    # Target page URL
    url = "https://en.wikipedia.org/wiki/Floating_point_operations_per_second#Computational_performance"

    # Headers to simulate a browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        # Send GET request
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Check for HTTP errors

        # Parse HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all wikitable tables
        tables = soup.find_all('table', class_='wikitable')

        target_table = None
        # Find the table containing headers with FP64, FP32, FP16
        for table in tables:
            headers_row = table.find('tr')
            if headers_row and 'FP64' in headers_row.text and 'FP32' in headers_row.text and 'FP16' in headers_row.text:
                target_table = table
                break

        if not target_table:
            print("Table not found!")
            return

        # Extract data
        rows = target_table.find_all('tr')

        # Prepare list to store results
        data = []

        # Variable to store current manufacturer/category
        current_category = ""

        for row in rows:
            # Skip the header row
            if 'FP64' in row.text and 'FP32' in row.text and 'FP16' in row.text:
                continue

            cells = row.find_all(['td', 'th'])
            if not cells:
                continue

            # Check if the row is a category (e.g., "Intel CPU")
            if len(cells) == 1:
                category_cell = cells[0]
                # Remove [edit] notes from the category text
                current_category = re.sub(r'\[.*?\]', '', category_cell.text).strip()
                continue

            # Regular row with data
            if len(cells) >= 4:
                # Extract and clean text from cells
                microarchitecture = cells[0].text.strip()
                isa = cells[1].text.strip() if len(cells) > 1 else ""
                fp64 = cells[2].text.strip() if len(cells) > 2 else ""
                fp32 = cells[3].text.strip() if len(cells) > 3 else ""
                fp16 = cells[4].text.strip() if len(cells) > 4 else ""

                # Remove notes like [12], [13], etc.
                microarchitecture = re.sub(r'\[.*?\]', '', microarchitecture)
                isa = re.sub(r'\[.*?\]', '', isa)
                fp64 = re.sub(r'\[.*?\]', '', fp64)
                fp32 = re.sub(r'\[.*?\]', '', fp32)
                fp16 = re.sub(r'\[.*?\]', '', fp16)

                # Replace empty strings and "?" with "N/A"
                fp64 = fp64 if fp64 and fp64 != '?' else 'N/A'
                fp32 = fp32 if fp32 and fp32 != '?' else 'N/A'
                fp16 = fp16 if fp16 and fp16 != '?' else 'N/A'

                # Add entry with category
                data.append([
                    current_category,
                    microarchitecture,
                    isa,
                    fp64,
                    fp32,
                    fp16
                ])

        # Save to CSV file
        output_file = '../02_raw/wiki_fp32.csv'
        with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            # Write headers
            writer.writerow(['Category', 'Microarchitecture', 'ISA', 'FP64', 'FP32', 'FP16'])
            # Write data
            writer.writerows(data)

        print(f"Data successfully saved to file: {output_file}")
        print(f"Total entries: {len(data)}")

        # Print first few rows for verification
        print("\nFirst 5 entries:")
        for row in data[:5]:
            print(row)

    except requests.exceptions.RequestException as e:
        print(f"Error while requesting the page: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    parse_wiki_flops_table()